In [0]:
%run ./01.config

In [0]:
from pyspark.sql import functions as F
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)


class DailyDemandAggregator:
    
    def __init__(self, spark, source_table):
        self.spark = spark
        self.source_table = source_table
    
    def aggregate_and_save(self, target_table):
        try:
            logger.info(f"Loading data from {self.source_table}")
            silver = self.spark.table(self.source_table)
            
            daily_demand = silver.groupBy(
                F.col("txn_date").alias("transaction_date"),
                "product_name",
                "destination_city"
            ).agg(
                F.sum(F.col("demand_quantity").cast("double")).alias("total_demand"),
                F.avg(F.col("unit_price_usd").cast("double")).alias("avg_unit_price"),
                F.sum(F.col("available_inventory").cast("double")).alias("total_inventory"),
                F.count("*").alias("transaction_count")
            )
            
            logger.info(f"Saving to {target_table}")
            daily_demand.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
            
            result = self.spark.table(target_table)
            total_rows = result.count()
            
            logger.info(f"Done! Created {target_table} with {total_rows:,} rows")
            return result
            
        except Exception as e:
            logger.error(f"Error: {str(e)}")
            raise


aggregator = DailyDemandAggregator(spark, SILVER)
daily_demand = aggregator.aggregate_and_save(GOLD_AGG)